In [0]:
import requests
import json
import pandas as pd

In [0]:
tmdb_key = dbutils.secrets.get(scope='tmdb_keys', key='tmdb_key')

In [0]:
df = spark.read.table('workspace.bronze.recomendation_db_bronze')
df = df.toPandas()

In [0]:
# Criando a base de dados com informações básicas sobre o filme

headers = {
    "accept": "application/json",
    "Authorization": tmdb_key
}

info_movies = []

for movie_id in df['id']:

    url = f'https://api.themoviedb.org/3/movie/{movie_id}?language=pt-BR'
    response = requests.get(url, headers= headers)
    data = response.json()

    info_movies.append({
        "id": data['id'],
        'title': data['title'],
        'realese_year': data['release_date'],
        'poster_path': f'https://image.tmdb.org/t/p/w500{data['poster_path']}' if data["poster_path"] else None
        })


movies_info = pd.DataFrame(info_movies)

In [0]:
movies_info_spark = spark.createDataFrame(movies_info)
movies_info_spark.write.mode('overwrite').option("mergeSchema", "true").saveAsTable('workspace.bronze.movies_info_bronze')